# Laya V2.7 — PDelta3+Local32 Full Attention + AttentionCeNN Sliding Attention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V27_PDelta3_Local32_AttentionCeNN_Colab.ipynb)

V2.7 is a full-model hybrid conversion attempt built from the strongest V2.6 direction:

- **Full-attention layers** → **PDelta3-GDN2 + bidirectional Local32**. The learned global linear memory supplies full-sequence context; GDN2 supplies recurrent state; Local32 supplies exact query-dependent nearby retrieval.
- **Sliding-attention layers** → **AttentionCeNN V2.7**. It keeps pretrained Q/K/V, uses a contractive bidirectional CeNN state across the original local receptive field, and uses query-key-conditioned routing without a pairwise sliding softmax matrix.
- Every layer is trained against the **unchanged Laya teacher**, inserted temporarily, evaluated end-to-end, and **rolled back if it fails**.
- Full-attention conversion runs first while native sliding attention is still intact. Sliding conversion runs second.
- A short candidate-only end-to-end recovery is attempted before rejecting a locally good layer.
- A final low-LR global recovery jointly calibrates all accepted replacements.
- The official test split is used **only once at the end**.

This notebook does not claim all 28 layers will pass. A partial conversion is kept only when it preserves the teacher quality gates.


In [ ]:
#@title 1. Install
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"


In [ ]:
#@title 2. Imports, configuration, teacher/student
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from tinycenn_lm.decision_v25 import layer_kinds
from tinycenn_lm.decision_v27 import (
    SlidingAttentionCeNN,
    make_v27_replacement,
    v27_replacement_kind,
)
from tinycenn_lm.laya_lab.pdelta import PDelta3GDN2CLVRAttention
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence, collate_items,
    normalize_question,
)
from tinycenn_lm.decision_training import (
    require_finite, guarded_step, forward_with_markers,
    marker_alignment_loss, decision_distill_loss,
)

SEED = 2026
SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}

TRAIN_CASES = 600 #@param {type:"integer"}
DEV_CASES = 100 #@param {type:"integer"}
MAX_LEN = 512 #@param {type:"integer"}
BATCH = 2 #@param {type:"integer"}

# Full-attention PDelta3 + Local32. F=96 is the V2.6 layer-9 winner setting.
FULL_FEATURE_DIM = 96 #@param {type:"integer"}
FULL_LOCAL_WINDOW = 32 #@param {type:"integer"}
FULL_CHUNK_SIZE = 32 #@param {type:"integer"}
FULL_LOCAL_STEPS = 2000 #@param {type:"integer"}

# Sliding AttentionCeNN. Two recurrent steps with window 128 give an
# approximate +/-64-token CeNN receptive field.
SLIDING_WINDOW = 128 #@param {type:"integer"}
SLIDING_STEPS = 2 #@param {type:"integer"}
SLIDING_LOCAL_STEPS = 1400 #@param {type:"integer"}

TRAIN_CACHE_BATCHES = 48 #@param {type:"integer"}
PROBE_CACHE_BATCHES = 10 #@param {type:"integer"}
LOCAL_CHECK_EVERY = 100 #@param {type:"integer"}

# Candidate-only decision recovery before rollback.
FULL_E2E_STEPS = 180 #@param {type:"integer"}
SLIDING_E2E_STEPS = 120 #@param {type:"integer"}
E2E_CHECK_EVERY = 30 #@param {type:"integer"}

# Joint low-LR recovery after both conversion stages.
RUN_GLOBAL_RECOVERY = True #@param {type:"boolean"}
GLOBAL_RECOVERY_STEPS = 300 #@param {type:"integer"}
GLOBAL_CHECK_EVERY = 50 #@param {type:"integer"}

# Strict cumulative gates. DEV_GATE_LIMIT counts decisions, not source cases.
DEV_GATE_LIMIT = 500 #@param {type:"integer"}
MIN_TEACHER_AGREEMENT = 0.95 #@param {type:"number"}
MAX_MEAN_JS = 0.02 #@param {type:"number"}
MAX_GOLD_ACCURACY_DROP = 0.02 #@param {type:"number"}
MAX_PER_LAYER_AGREEMENT_DROP = 0.015 #@param {type:"number"}

# Local attention-transfer gates.
FULL_MAX_NMSE = 0.20
FULL_MIN_COSINE = 0.90
SLIDING_MAX_NMSE = 0.20
SLIDING_MIN_COSINE = 0.90

OUTPUT_DIR = Path("/content/Laya_V27_PDelta_Local32_AttentionCeNN")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROGRESS_PATH = OUTPUT_DIR / "v27_progress.pt"
REPORT_PATH = OUTPUT_DIR / "v27_report.json"

RESUME = True #@param {type:"boolean"}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("V2.7 requires a CUDA GPU.")

device = torch.device("cuda")
amp_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported(including_emulation=False)
    else torch.float16
)

source_dir = Path(snapshot_download(
    SOURCE_MODEL,
    allow_patterns=[
        "rl_agent_config.json",
        "model.safetensors",
        "encoder/*",
        "tokenizer/*",
    ],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
student, _ = build_checkpoint_model(source_dir)
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))

teacher.to(device).eval().requires_grad_(False)
student.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile = False
student.encoder.config.reference_compile = False

kinds = layer_kinds(teacher.encoder)
full_layers = [i for i, k in enumerate(kinds) if k == "full_attention"]
sliding_layers = [i for i, k in enumerate(kinds) if k == "sliding_attention"]

print("device:", torch.cuda.get_device_name(0))
print("amp:", amp_dtype)
print("full layers:", full_layers)
print("sliding layers:", sliding_layers)
print("V2.7 full:", f"PDelta3 F={FULL_FEATURE_DIM} + Local{FULL_LOCAL_WINDOW}")
print("V2.7 sliding:", f"AttentionCeNN window={SLIDING_WINDOW}, steps={SLIDING_STEPS}")


In [ ]:
#@title 3. Leak-free train/dev data
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon = [_canon_label(k) for k in labels]
    if gold_label not in canon:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer, state, q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

all_train_rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
split_rng = random.Random(SEED)
split_rng.shuffle(all_train_rows)
assert TRAIN_CASES + DEV_CASES <= len(all_train_rows)

train_rows = all_train_rows[:TRAIN_CASES]
dev_rows = all_train_rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
dev_items = make_items_from_rows(dev_rows, SEED + 1, True)

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

print("train/dev cases:", len(train_rows), len(dev_rows))
print("train/dev decisions:", len(train_items), len(dev_items))
print("official test is deferred until the final cell")


In [ ]:
#@title 4. Teacher I/O, local metrics, and whole-model gates
def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        got["x"] = args[0].detach()
        got["pos"] = None if kwargs.get("position_embeddings") is None else tuple(
            z.detach() for z in kwargs["position_embeddings"]
        )
        got["mask"] = None if kwargs.get("attention_mask") is None else kwargs["attention_mask"].detach()
    def post(mod, args, kwargs, out):
        got["y"] = out[0].detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

@torch.no_grad()
def local_metrics(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    for entry in cache:
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            pred, _ = repl(
                entry["x"],
                position_embeddings=entry["pos"],
                attention_mask=entry["mask"],
            )
        require_finite(pred, "V2.7 local probe")
        valid = entry["valid"]
        m = valid[:, :, None].float()
        diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
        power_sum += float(((entry["y"].float() ** 2) * m).sum())
        cos_sum += float(F.cosine_similarity(
            pred.float()[valid],
            entry["y"].float()[valid],
            dim=-1,
        ).sum())
        tokens += int(valid.sum())
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    return {
        "nmse": nmse,
        "cosine": cosine,
        "score": nmse + 1.10 * (1.0 - cosine),
        "tokens": tokens,
    }

@torch.no_grad()
def validate_pair(model, items=dev_items, limit=None):
    teacher.eval()
    model.eval()
    limit = len(items) if limit is None else min(int(limit), len(items))
    total = agree = teacher_gold = student_gold = 0
    js = []

    for off in range(0, limit, BATCH):
        selected = items[off:min(off + BATCH, limit)]
        b = direct_batch(selected)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            tl, _ = teacher(**b)
            sl, _ = model(**b)
        require_finite(tl, "teacher validation logits")
        require_finite(sl, "student validation logits")

        mask = b["marker_mask"].bool()
        tm = tl.float().masked_fill(~mask, -1e9)
        sm = sl.float().masked_fill(~mask, -1e9)
        tp = torch.softmax(tm, -1)
        sp = torch.softmax(sm, -1)
        tpred, spred = tm.argmax(-1), sm.argmax(-1)
        gold = torch.tensor(
            [item["gold_index"] for item in selected],
            device=device,
            dtype=torch.long,
        )

        agree += int((tpred == spred).sum())
        teacher_gold += int((tpred == gold).sum())
        student_gold += int((spred == gold).sum())
        total += len(selected)

        mid = 0.5 * (tp + sp)
        j = 0.5 * (
            tp * (tp.clamp_min(1e-8).log() - mid.clamp_min(1e-8).log())
        ).sum(-1)
        j += 0.5 * (
            sp * (sp.clamp_min(1e-8).log() - mid.clamp_min(1e-8).log())
        ).sum(-1)
        js.extend(j.cpu().tolist())

    return {
        "decisions": int(total),
        "agreement": agree / max(1, total),
        "mean_js": float(np.mean(js)),
        "teacher_gold_accuracy": teacher_gold / max(1, total),
        "student_gold_accuracy": student_gold / max(1, total),
        "accuracy_drop": (teacher_gold - student_gold) / max(1, total),
    }

def cumulative_pass(baseline, candidate):
    checks = {
        "teacher_agreement": candidate["agreement"] >= MIN_TEACHER_AGREEMENT,
        "mean_js": candidate["mean_js"] <= MAX_MEAN_JS,
        "gold_accuracy_drop": candidate["accuracy_drop"] <= MAX_GOLD_ACCURACY_DROP,
        "incremental_agreement_drop": (
            baseline["agreement"] - candidate["agreement"]
            <= MAX_PER_LAYER_AGREEMENT_DROP
        ),
    }
    return all(checks.values()), checks

def dev_rank(m):
    return (
        m["mean_js"]
        + 0.50 * (1.0 - m["agreement"])
        + 0.50 * max(0.0, m["accuracy_drop"])
    )


In [ ]:
#@title 5. Candidate optimizer, local transfer, and end-to-end recovery
def state_cpu(module):
    return {k: v.detach().cpu().clone() for k, v in module.state_dict().items()}

def _unique(params):
    seen, out = set(), []
    for p in params:
        if p.requires_grad and id(p) not in seen:
            seen.add(id(p))
            out.append(p)
    return out

def candidate_groups(repl, kind, phase):
    # FP32 master weights improve the low-LR output calibration.
    core, output = [], []
    for name, p in repl.named_parameters():
        if name.startswith("Wqkv.") or name.startswith("out_drop."):
            p.requires_grad = False
        elif name.startswith("Wo."):
            p.requires_grad = True
            p.data = p.data.float()
            output.append(p)
        else:
            p.requires_grad = True
            if p.is_floating_point():
                p.data = p.data.float()
            core.append(p)

    core, output = _unique(core), _unique(output)
    if phase == "local":
        if kind == "full_attention":
            core_lr = (1.0e-2, 1.0e-3)
        else:
            core_lr = (5.0e-3, 5.0e-4)
        out_lr = (core_lr[0] * 0.10, core_lr[1] * 0.10)
    else:
        if kind == "full_attention":
            core_lr = (2.0e-4, 4.0e-5)
        else:
            core_lr = (1.5e-4, 3.0e-5)
        out_lr = (core_lr[0] * 0.10, core_lr[1] * 0.10)

    groups = [
        {"params": core, "lr": core_lr[0], "_start": core_lr[0], "_end": core_lr[1]},
    ]
    if output:
        groups.append({
            "params": output, "lr": out_lr[0],
            "_start": out_lr[0], "_end": out_lr[1],
        })
    return groups, _unique(core + output)

def cosine_lr(step, total, start, end):
    if total <= 1:
        return float(end)
    x = min(1.0, max(0.0, step / float(total - 1)))
    return float(end + 0.5 * (start - end) * (1.0 + math.cos(math.pi * x)))

def fit_local(layer_idx, kind, repl, train_cache, probe_cache, steps):
    groups, clip_params = candidate_groups(repl, kind, "local")
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.95), weight_decay=2e-4)
    scaler = torch.amp.GradScaler(
        "cuda", enabled=amp_dtype == torch.float16, init_scale=128.0
    )

    best = local_metrics(repl, probe_cache)
    best_state = state_cpu(repl)
    best_step = 0
    updates = attempts = 0
    max_attempts = int(steps) + 256

    print(
        f"  local start: NMSE={best['nmse']:.4f} "
        f"cos={best['cosine']:.4f}"
    )

    while updates < steps:
        if attempts >= max_attempts:
            raise RuntimeError(
                f"layer {layer_idx}: too many AMP skips "
                f"({updates}/{steps} successful)"
            )
        for group in opt.param_groups:
            group["lr"] = cosine_lr(
                updates, steps, group["_start"], group["_end"]
            )

        entry = train_cache[attempts % len(train_cache)]
        repl.train()
        if hasattr(repl, "out_drop"):
            repl.out_drop.eval()
        opt.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            pred, _ = repl(
                entry["x"],
                position_embeddings=entry["pos"],
                attention_mask=entry["mask"],
            )
            valid = entry["valid"]
            m = valid[:, :, None].float()
            mse = (
                ((pred.float() - entry["y"].float()) ** 2) * m
            ).sum() / (m.sum() * pred.shape[-1]).clamp_min(1)
            power = (
                (entry["y"].float() ** 2) * m
            ).sum() / (m.sum() * pred.shape[-1]).clamp_min(1)
            nmse = mse / power.clamp_min(1e-8)
            cosine = F.cosine_similarity(
                pred.float()[valid],
                entry["y"].float()[valid],
                dim=-1,
            ).mean()
            loss = nmse + 1.10 * (1.0 - cosine)

        updated = guarded_step(
            loss, opt, scaler,
            [(clip_params, 1.0)],
            context=f"V2.7 layer {layer_idx} local {updates + 1}",
        )
        attempts += 1
        if not updated:
            continue
        updates += 1

        if (
            updates == 1
            or updates % LOCAL_CHECK_EVERY == 0
            or updates == steps
        ):
            probe = local_metrics(repl, probe_cache)
            if probe["score"] < best["score"]:
                best, best_state, best_step = probe, state_cpu(repl), updates
            print(
                f"    {updates:04d}/{steps} "
                f"NMSE={probe['nmse']:.4f} cos={probe['cosine']:.4f} "
                f"best={best['score']:.4f}"
            )

    repl.load_state_dict(best_state, strict=True)
    repl.eval()
    return {
        "local": best,
        "best_step": int(best_step),
        "updates": int(updates),
        "attempts": int(attempts),
    }

def e2e_recover(layer_idx, kind, repl, steps):
    groups, clip_params = candidate_groups(repl, kind, "e2e")
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.95), weight_decay=1e-4)
    scaler = torch.amp.GradScaler(
        "cuda", enabled=amp_dtype == torch.float16, init_scale=128.0
    )

    # Keep the full model deterministic. For PDelta this also means its recurrent
    # scan is not gradient-detached every chunk, so recovery gets full BPTT.
    student.eval()
    baseline = validate_pair(student, dev_items, DEV_GATE_LIMIT)
    best = baseline
    best_score = dev_rank(baseline)
    best_state = state_cpu(repl)
    best_step = 0
    updates = attempts = 0

    print(
        f"  E2E start: agree={baseline['agreement']:.4f} "
        f"JS={baseline['mean_js']:.5f} "
        f"gold_drop={baseline['accuracy_drop']:+.4f}"
    )

    while updates < steps:
        for group in opt.param_groups:
            group["lr"] = cosine_lr(
                updates, steps, group["_start"], group["_end"]
            )
        b = batch(train_items, attempts * BATCH + layer_idx * 53, BATCH)
        opt.zero_grad(set_to_none=True)

        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            tl, ta, tf = forward_with_markers(teacher, b)

        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            sl, sa, sf = forward_with_markers(student, b)
            loss = decision_distill_loss(
                sl, tl, b["marker_mask"].bool(), sa, ta,
                temperature=1.25,
                rank_margin=0.15,
                rank_weight=0.15,
            )
            loss = loss + 0.05 * marker_alignment_loss(
                sf, tf, b["marker_mask"].bool()
            )

        updated = guarded_step(
            loss, opt, scaler,
            [(clip_params, 0.75)],
            context=f"V2.7 layer {layer_idx} E2E {updates + 1}",
        )
        attempts += 1
        if not updated:
            continue
        updates += 1

        if (
            updates == 1
            or updates % E2E_CHECK_EVERY == 0
            or updates == steps
        ):
            metrics = validate_pair(student, dev_items, DEV_GATE_LIMIT)
            score = dev_rank(metrics)
            if score < best_score:
                best = metrics
                best_score = score
                best_state = state_cpu(repl)
                best_step = updates
            print(
                f"    E2E {updates:03d}/{steps} "
                f"agree={metrics['agreement']:.4f} "
                f"JS={metrics['mean_js']:.5f} "
                f"gold_drop={metrics['accuracy_drop']:+.4f}"
            )

    repl.load_state_dict(best_state, strict=True)
    repl.eval()
    student.eval().requires_grad_(False)
    return {
        "baseline": baseline,
        "selected": best,
        "best_step": int(best_step),
        "updates": int(updates),
        "attempts": int(attempts),
    }


In [ ]:
#@title 6. Build replacements, progress save/resume, and GPU preflight
def build_replacement(layer_idx, source_attn=None):
    if source_attn is None:
        source_attn = teacher.encoder.layers[layer_idx].attn
    return make_v27_replacement(
        source_attn,
        kinds[layer_idx],
        full_feature_dim=FULL_FEATURE_DIM,
        full_local_window=FULL_LOCAL_WINDOW,
        full_chunk_size=FULL_CHUNK_SIZE,
        sliding_window=SLIDING_WINDOW,
        sliding_steps=SLIDING_STEPS,
    ).to(device)

def replacement_config():
    return {
        "source_model": SOURCE_MODEL,
        "full_feature_dim": FULL_FEATURE_DIM,
        "full_local_window": FULL_LOCAL_WINDOW,
        "full_chunk_size": FULL_CHUNK_SIZE,
        "sliding_window": SLIDING_WINDOW,
        "sliding_steps": SLIDING_STEPS,
    }

accepted = {}
history = []
completed_layers = set()

def save_progress():
    payload = {
        "version": "2.7",
        "config": replacement_config(),
        "accepted": {
            int(i): {
                "kind": rec["kind"],
                "replacement": rec["replacement"],
            }
            for i, rec in accepted.items()
        },
        "completed_layers": sorted(int(i) for i in completed_layers),
        "history": history,
        "state_dict": {
            k: v.detach().cpu()
            for k, v in student.state_dict().items()
        },
    }
    torch.save(payload, PROGRESS_PATH)

def restore_progress():
    global accepted, history, completed_layers
    if not (RESUME and PROGRESS_PATH.exists()):
        return False
    payload = torch.load(PROGRESS_PATH, map_location="cpu")
    if payload.get("version") != "2.7":
        raise ValueError("progress file is not V2.7")
    if payload.get("config") != replacement_config():
        raise ValueError("V2.7 progress configuration differs from this run")

    for key, rec in payload.get("accepted", {}).items():
        idx = int(key)
        student.encoder.layers[idx].attn = build_replacement(idx)
    student.load_state_dict(payload["state_dict"], strict=True)
    student.to(device).eval().requires_grad_(False)

    accepted = {int(k): dict(v) for k, v in payload.get("accepted", {}).items()}
    history = list(payload.get("history", []))
    completed_layers = set(int(i) for i in payload.get("completed_layers", []))
    print("Resumed completed layers:", sorted(completed_layers))
    print("Resumed accepted layers:", sorted(accepted))
    return True

restore_progress()

# Real forward/backward preflight for both replacement families.
for layer_idx in (full_layers[0], sliding_layers[0]):
    if layer_idx in completed_layers:
        continue
    b = batch(dev_items, layer_idx * BATCH, min(BATCH, 2))
    target = attn_io(teacher, layer_idx, b)
    repl = build_replacement(layer_idx)
    repl.train()
    if hasattr(repl, "out_drop"):
        repl.out_drop.eval()

    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        pred, _ = repl(
            target["x"],
            position_embeddings=target["pos"],
            attention_mask=target["mask"],
        )
        valid = b["attention_mask"].bool()
        m = valid[:, :, None].float()
        loss = (
            ((pred.float() - target["y"].float()) ** 2) * m
        ).sum() / (m.sum() * pred.shape[-1]).clamp_min(1)

    require_finite(pred, "V2.7 preflight output")
    require_finite(loss, "V2.7 preflight loss")
    params = [p for p in repl.parameters() if p.requires_grad]
    grads = torch.autograd.grad(loss, params, allow_unused=True)
    finite = [g for g in grads if g is not None and torch.isfinite(g).all()]
    if not finite or sum(float(g.abs().sum()) for g in finite) <= 0:
        raise RuntimeError(f"layer {layer_idx}: no finite non-zero preflight gradients")

    print(
        f"preflight layer {layer_idx:02d} {kinds[layer_idx]:<18} "
        f"{v27_replacement_kind(repl):<18} PASS "
        f"loss={float(loss.detach()):.5f}"
    )
    del repl, pred, loss, grads, finite, target, b
    torch.cuda.empty_cache()


In [ ]:
#@title 7. Stage A — replace ALL full-attention layers with PDelta3 + Local32
def local_gate(kind, metrics):
    if kind == "full_attention":
        return (
            metrics["nmse"] <= FULL_MAX_NMSE
            and metrics["cosine"] >= FULL_MIN_COSINE
        )
    return (
        metrics["nmse"] <= SLIDING_MAX_NMSE
        and metrics["cosine"] >= SLIDING_MIN_COSINE
    )

def attempt_layer(layer_idx):
    kind = kinds[layer_idx]
    if layer_idx in completed_layers:
        print(f"layer {layer_idx:02d}: already completed, skipping")
        return

    print("\n" + "=" * 104)
    print(f"V2.7 {kind.upper()} LAYER {layer_idx}")
    print("=" * 104)

    baseline = validate_pair(student, dev_items, DEV_GATE_LIMIT)
    print("cumulative baseline:", baseline)

    train_cache = cache_teacher_io(
        layer_idx, train_items, TRAIN_CACHE_BATCHES,
        offset=layer_idx * 17,
    )
    probe_cache = cache_teacher_io(
        layer_idx, dev_items, PROBE_CACHE_BATCHES,
        offset=layer_idx * 11,
    )
    repl = build_replacement(layer_idx)
    steps = FULL_LOCAL_STEPS if kind == "full_attention" else SLIDING_LOCAL_STEPS

    local_report = fit_local(
        layer_idx, kind, repl, train_cache, probe_cache, steps
    )
    local = local_report["local"]
    local_ok = local_gate(kind, local)

    original = student.encoder.layers[layer_idx].attn
    student.encoder.layers[layer_idx].attn = repl
    student.eval().requires_grad_(False)

    candidate = validate_pair(student, dev_items, DEV_GATE_LIMIT)
    cumulative_ok, checks = cumulative_pass(baseline, candidate)

    print(json.dumps({
        "layer": layer_idx,
        "kind": kind,
        "replacement": v27_replacement_kind(repl),
        "local": local,
        "local_ok": local_ok,
        "cumulative": candidate,
        "checks": checks,
    }, indent=2))

    recovery = None
    # Recover only candidates that are locally credible. This prevents decision
    # distillation from hiding a structurally poor attention approximation.
    recoverable = (
        local["nmse"] <= 0.25
        and local["cosine"] >= 0.88
    )
    if not cumulative_ok and recoverable:
        print("Local approximation is credible; trying candidate-only E2E recovery...")
        # Re-enable only replacement parameters inside e2e_recover.
        recovery_steps = FULL_E2E_STEPS if kind == "full_attention" else SLIDING_E2E_STEPS
        recovery = e2e_recover(layer_idx, kind, repl, recovery_steps)
        candidate = validate_pair(student, dev_items, DEV_GATE_LIMIT)
        cumulative_ok, checks = cumulative_pass(baseline, candidate)
        local = local_metrics(repl, probe_cache)
        local_ok = local_gate(kind, local)
        print("post-recovery local:", local)
        print("post-recovery cumulative:", candidate)
        print("post-recovery checks:", checks)

    accepted_now = bool(local_ok and cumulative_ok)
    rec = {
        "layer": int(layer_idx),
        "kind": kind,
        "replacement": v27_replacement_kind(repl),
        "baseline": baseline,
        "local_training": local_report,
        "local_final": local,
        "cumulative": candidate,
        "checks": checks,
        "recovery": recovery,
        "accepted": accepted_now,
    }

    if accepted_now:
        repl.eval().requires_grad_(False)
        accepted[layer_idx] = {
            "kind": kind,
            "replacement": v27_replacement_kind(repl),
        }
        print(
            f"✅ ACCEPTED layer {layer_idx}: "
            f"{v27_replacement_kind(repl)} | "
            f"agreement={candidate['agreement']:.4f}"
        )
    else:
        student.encoder.layers[layer_idx].attn = original
        student.eval().requires_grad_(False)
        print(
            f"❌ REJECTED + ROLLED BACK layer {layer_idx} | "
            f"local_ok={local_ok}, cumulative_ok={cumulative_ok}"
        )

    history.append(rec)
    completed_layers.add(layer_idx)
    save_progress()

    del train_cache, probe_cache, repl
    torch.cuda.empty_cache()

# Preserve layer order so later candidates see the real cumulative upstream model.
for layer_idx in full_layers:
    attempt_layer(layer_idx)

print("\nStage A accepted full layers:",
      [i for i in full_layers if i in accepted])
print("Stage A remaining native full layers:",
      [i for i in full_layers if i not in accepted])


In [ ]:
#@title 8. Stage B — replace ALL sliding-attention layers with AttentionCeNN
for layer_idx in sliding_layers:
    attempt_layer(layer_idx)

accepted_full = [i for i in full_layers if i in accepted]
accepted_sliding = [i for i in sliding_layers if i in accepted]
remaining_full = [i for i in full_layers if i not in accepted]
remaining_sliding = [i for i in sliding_layers if i not in accepted]

print("\nAccepted full PDelta3+Local32:", accepted_full)
print("Accepted sliding AttentionCeNN:", accepted_sliding)
print("Remaining native full attention:", remaining_full)
print("Remaining native sliding attention:", remaining_sliding)
print(
    "Conversion coverage:",
    f"{len(accepted)}/{len(kinds)} = {100*len(accepted)/len(kinds):.1f}%"
)


In [ ]:
#@title 9. Optional global low-LR recovery of all accepted replacements
def replacement_modules():
    return [
        (i, student.encoder.layers[i].attn)
        for i in sorted(accepted)
    ]

def snapshot_replacements():
    return {
        int(i): state_cpu(module)
        for i, module in replacement_modules()
    }

def restore_replacements(snapshot):
    for i, module in replacement_modules():
        module.load_state_dict(snapshot[int(i)], strict=True)
        module.eval()

global_recovery = None

if RUN_GLOBAL_RECOVERY and accepted:
    core, output = [], []
    for i, module in replacement_modules():
        for name, p in module.named_parameters():
            if name.startswith("Wqkv.") or name.startswith("out_drop."):
                p.requires_grad = False
            elif name.startswith("Wo."):
                p.requires_grad = True
                output.append(p)
            else:
                p.requires_grad = True
                core.append(p)
    core, output = _unique(core), _unique(output)

    groups = [
        {"params": core, "lr": 1.0e-4, "_start": 1.0e-4, "_end": 2.0e-5},
    ]
    if output:
        groups.append({
            "params": output, "lr": 1.0e-5,
            "_start": 1.0e-5, "_end": 2.0e-6,
        })

    opt = torch.optim.AdamW(groups, betas=(0.9, 0.95), weight_decay=1e-4)
    scaler = torch.amp.GradScaler(
        "cuda", enabled=amp_dtype == torch.float16, init_scale=128.0
    )
    clip_params = _unique(core + output)

    student.eval()
    baseline = validate_pair(student, dev_items, DEV_GATE_LIMIT)
    best = baseline
    best_score = dev_rank(baseline)
    best_state = snapshot_replacements()
    best_step = 0
    updates = attempts = 0

    print("global recovery baseline:", baseline)

    while updates < GLOBAL_RECOVERY_STEPS:
        for group in opt.param_groups:
            group["lr"] = cosine_lr(
                updates, GLOBAL_RECOVERY_STEPS,
                group["_start"], group["_end"],
            )

        # Global recovery uses batch 1 to keep full-BPTT PDelta memory bounded.
        b = batch(train_items, attempts + 7000, 1)
        opt.zero_grad(set_to_none=True)

        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            tl, ta, tf = forward_with_markers(teacher, b)

        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            sl, sa, sf = forward_with_markers(student, b)
            loss = decision_distill_loss(
                sl, tl, b["marker_mask"].bool(), sa, ta,
                temperature=1.25,
                rank_margin=0.15,
                rank_weight=0.15,
            )
            loss = loss + 0.05 * marker_alignment_loss(
                sf, tf, b["marker_mask"].bool()
            )

        updated = guarded_step(
            loss, opt, scaler,
            [(clip_params, 0.50)],
            context=f"V2.7 global recovery {updates + 1}",
        )
        attempts += 1
        if not updated:
            continue
        updates += 1

        if (
            updates == 1
            or updates % GLOBAL_CHECK_EVERY == 0
            or updates == GLOBAL_RECOVERY_STEPS
        ):
            metrics = validate_pair(student, dev_items, DEV_GATE_LIMIT)
            score = dev_rank(metrics)
            if score < best_score:
                best, best_score = metrics, score
                best_state = snapshot_replacements()
                best_step = updates
            print(
                f"  global {updates:03d}/{GLOBAL_RECOVERY_STEPS} "
                f"agree={metrics['agreement']:.4f} "
                f"JS={metrics['mean_js']:.5f} "
                f"gold_drop={metrics['accuracy_drop']:+.4f}"
            )

    restore_replacements(best_state)
    student.eval().requires_grad_(False)
    global_recovery = {
        "baseline": baseline,
        "selected": best,
        "best_step": int(best_step),
        "updates": int(updates),
        "attempts": int(attempts),
    }
    print("selected global recovery:", global_recovery)

    save_progress()
else:
    print("Global recovery skipped.")


In [ ]:
#@title 10. Final untouched official-test evaluation, latency, report, checkpoint
test_rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="test"))
test_items = make_items_from_rows(test_rows, SEED + 99, False)

dev_final = validate_pair(student, dev_items, len(dev_items))
test_final = validate_pair(student, test_items, len(test_items))

def benchmark_model(model, selected, warmup=5, repeats=20):
    b = direct_batch(selected)
    model.eval()
    for _ in range(warmup):
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    torch.cuda.synchronize()
    values = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
        torch.cuda.synchronize()
        values.append((time.perf_counter() - t0) * 1000.0)
    return {
        "median_ms": float(np.median(values)),
        "p95_ms": float(np.percentile(values, 95)),
        "mean_ms": float(np.mean(values)),
        "repeats": repeats,
        "batch": len(selected),
    }

latency_batch = dev_items[:1]
teacher_latency = benchmark_model(teacher, latency_batch)
student_latency = benchmark_model(student, latency_batch)
latency = {
    "teacher": teacher_latency,
    "student": student_latency,
    "speedup": teacher_latency["median_ms"] / max(student_latency["median_ms"], 1e-9),
}

audit = {
    "accepted_full": [i for i in full_layers if i in accepted],
    "accepted_sliding": [i for i in sliding_layers if i in accepted],
    "remaining_full": [i for i in full_layers if i not in accepted],
    "remaining_sliding": [i for i in sliding_layers if i not in accepted],
}
audit["all_full_replaced"] = not audit["remaining_full"]
audit["all_sliding_replaced"] = not audit["remaining_sliding"]
audit["all_attention_replaced"] = (
    audit["all_full_replaced"] and audit["all_sliding_replaced"]
)

report = {
    "version": "2.7",
    "source_model": SOURCE_MODEL,
    "architecture": {
        "full_attention": {
            "replacement": "pdelta3_gdn2_clvr_local32",
            "feature_dim": FULL_FEATURE_DIM,
            "effective_global_feature_dim": 2 * FULL_FEATURE_DIM,
            "local_window": FULL_LOCAL_WINDOW,
            "chunk_size_local_training": FULL_CHUNK_SIZE,
            "candidate_recovery_full_bptt": True,
        },
        "sliding_attention": {
            "replacement": "sliding_attention_cenn_v27",
            "window": SLIDING_WINDOW,
            "steps": SLIDING_STEPS,
            "query_key_conditioning": True,
            "exact_pairwise_attention": False,
        },
    },
    "gates": {
        "min_teacher_agreement": MIN_TEACHER_AGREEMENT,
        "max_mean_js": MAX_MEAN_JS,
        "max_gold_accuracy_drop": MAX_GOLD_ACCURACY_DROP,
        "max_per_layer_agreement_drop": MAX_PER_LAYER_AGREEMENT_DROP,
        "full_max_nmse": FULL_MAX_NMSE,
        "full_min_cosine": FULL_MIN_COSINE,
        "sliding_max_nmse": SLIDING_MAX_NMSE,
        "sliding_min_cosine": SLIDING_MIN_COSINE,
    },
    "audit": audit,
    "accepted": accepted,
    "history": history,
    "global_recovery": global_recovery,
    "dev_final": dev_final,
    "official_test_final": test_final,
    "latency": latency,
    "official_test_used_for_selection": False,
}
REPORT_PATH.write_text(
    json.dumps(report, indent=2, allow_nan=False),
    encoding="utf-8",
)

checkpoint_path = OUTPUT_DIR / "v27_final.pt"
torch.save({
    "version": "2.7",
    "config": replacement_config(),
    "accepted": accepted,
    "state_dict": {
        k: v.detach().cpu()
        for k, v in student.state_dict().items()
    },
    "report": report,
}, checkpoint_path)

print("\n" + "=" * 104)
print("V2.7 FINAL")
print("=" * 104)
print("coverage:", f"{len(accepted)}/{len(kinds)}")
print("all full replaced:", audit["all_full_replaced"])
print("all sliding replaced:", audit["all_sliding_replaced"])
print("all attention replaced:", audit["all_attention_replaced"])
print("remaining full:", audit["remaining_full"])
print("remaining sliding:", audit["remaining_sliding"])
print("\nDEV:", json.dumps(dev_final, indent=2))
print("\nOFFICIAL TEST:", json.dumps(test_final, indent=2))
print("\nLATENCY:", json.dumps(latency, indent=2))
print("\nReport:", REPORT_PATH)
print("Checkpoint:", checkpoint_path)


In [ ]:
#@title 11. Optional reload sanity check
RUN_RELOAD_CHECK = True #@param {type:"boolean"}

if RUN_RELOAD_CHECK:
    payload = torch.load(OUTPUT_DIR / "v27_final.pt", map_location="cpu")
    reloaded, _ = build_checkpoint_model(source_dir)
    reloaded.to(device).eval().requires_grad_(False)
    reloaded.encoder.config.reference_compile = False

    for key in payload["accepted"]:
        idx = int(key)
        reloaded.encoder.layers[idx].attn = make_v27_replacement(
            reloaded.encoder.layers[idx].attn,
            kinds[idx],
            full_feature_dim=FULL_FEATURE_DIM,
            full_local_window=FULL_LOCAL_WINDOW,
            full_chunk_size=FULL_CHUNK_SIZE,
            sliding_window=SLIDING_WINDOW,
            sliding_steps=SLIDING_STEPS,
        ).to(device)

    reloaded.load_state_dict(payload["state_dict"], strict=True)
    reloaded.eval().requires_grad_(False)

    sanity_items = dev_items[:min(32, len(dev_items))]
    a = validate_pair(student, sanity_items, len(sanity_items))
    b = validate_pair(reloaded, sanity_items, len(sanity_items))
    print("current :", a)
    print("reloaded:", b)

    if abs(a["agreement"] - b["agreement"]) > 1e-12:
        raise RuntimeError("reload sanity check failed")
    print("✅ reload sanity check passed")
